In [3]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Download the dataset
cdc_diabetes = fetch_ucirepo(id=891)

# Seperate features and target
x = cdc_diabetes.data.features
y = cdc_diabetes.data.targets

# Combine into one dataframe
df = pd.concat([x,y], axis=1)

# Save original downloaded data
df.to_csv("../data/raw/diabetes_health_indicators.csv", index=False)

In [2]:
# Check to make sure dataset was downloaded properly
print("Dataset dimensions:", df.shape)

# Display the first few records
display(df.head())

# Names of all the columns
print("\nColumn names:")
print(df.columns.tolist())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum().sum())

# Distribution of Diabetes 0 and 1
print("\nTarget distribution:")
print(df["Diabetes_binary"].value_counts())

# Check duplicates
print("\nDuplicates:")
print(df.duplicated().sum())

Dataset dimensions: (253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0



Column names:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income', 'Diabetes_binary']

Missing values:
0

Target distribution:
Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64

Duplicates:
24206


In [ ]:
# Make the dataframes for 4 social econoimic groups. We wanna check if there is sufficient data in each of these subgroups to be able train good models.

# Age groups
# Note: UCI uses these categories for age -> 13-level age category
#      1 = 18-24, 2 = 25-29, 3 = 30-34 
#      4 = 35-39, 5 = 40-44, etc. up to 13 = 80 or older

df["Age_Groups"] = pd.cut(
    df["Age"],
    bins=[0, 4, 8, 13],                                                     # Here our bins represent Younger Adults (18-39), Middle Age (40-59), Older (60+)
    labels=["Younger Adult", "Middle Age", "Older"],
)

# Income Groups
# Note: UCI uses these categories for income -> 8-level income category
#      1 = <$10,000, 2 = $10,000-$15,000, 3 = $15,000-$20,000 
#      4 = $20,000-$25,000, 5 = $25,000-$30,000, etc. up to 8 = $75,000 or more

df["Income_Groups"] = pd.cut(
    df["Income"],
    bins=[0, 3, 6, 8],                                                       # Here our bins represent Lower Income (<$10,000-$20,000), Middle Income ($20,000-$35,000), Higher Income ($35,000+)
    labels=["Lower Income", "Middle Income", "Higher Income"],
)


# Education Groups
# Note: UCI uses these categories for education -> 6-level education category
#      1 = No school or kindergarden, 2 = Grades 1-8, 3 = Grades 9-11 
#      4 = Grade 12 or GED (High school graduate), 5 = College 1-3 years, 6 = College >4 years

df["Education_Groups"] = pd.cut(
    df["Education"],
    bins=[0, 3, 4, 6],                                                      # Here our bins represent <HS, HS Graduate, >HS 
    labels=["Below High School", "High School", "Some College or Higher"],
)


# Sex labels
df["Sex_Group"] = df["Sex"].map({
    0: "Female",
    1: "Male"
})

In [6]:
# Here we examine the size and diabetes distribution of each group

demographic_columns = [
    "Sex_Group",
    "Age_Groups",
    "Income_Groups",
    "Education_Groups"
]

for column in demographic_columns:
    print(f"\n {column} ")

    summary = df.groupby(column, observed=False)["Diabetes_binary"].agg(
        Total="count",                                                              # Total number of people in each group
        Diabetes_Cases="sum",                                                       # Number of people with prediabetes/diabetes
        Diabetes_Rate="mean",                                                       # Percentage of each group w/ diabetes                                                    
    )

    summary["Diabetes_Rate"] *= 100

    display(summary.round(2))



 Sex_Group 


,Total,Diabetes_Cases,Diabetes_Rate
Sex_Group,,,
Female,141974,18411,12.97
Male,111706,16935,15.16



 Age_Groups 


,Total,Diabetes_Cases,Diabetes_Rate
Age_Groups,,,
Younger Adult,38244,1158,3.03
Middle Age,93122,10144,10.89
Older,122314,24044,19.66



 Income_Groups 


,Total,Diabetes_Cases,Diabetes_Rate
Income_Groups,,,
Lower Income,37588,9037,24.04
Middle Income,82488,13849,16.79
Higher Income,133604,12460,9.33



 Education_Groups 


,Total,Diabetes_Cases,Diabetes_Rate
Education_Groups,,,
Below High School,13695,3526,25.75
High School,62750,11066,17.64
Some College or Higher,177235,20754,11.71


From these results, we can see that these 4 social demographics have a good distribution and number of observations. All 4 groups have enough data to and interesting differences within subgroup diabetes rates to motivate the investigation of each independently. Furthermore, We continue by cleaning any unecessary data from the dataset.

In [ ]:
# Check for missing values
print("Missing values:")
display(df.isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check data types
print("\nData types:")
display(df.dtypes)

Missing values:


HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
Diabetes_binary         0
Age_Groups              0
Income_Groups           0
Education_Groups        0
Sex_Group               0
dtype: int64


Duplicate rows: 24206

Data types:


HighBP                     int64
HighChol                   int64
CholCheck                  int64
BMI                        int64
Smoker                     int64
Stroke                     int64
HeartDiseaseorAttack       int64
PhysActivity               int64
Fruits                     int64
Veggies                    int64
HvyAlcoholConsump          int64
AnyHealthcare              int64
NoDocbcCost                int64
GenHlth                    int64
MentHlth                   int64
PhysHlth                   int64
DiffWalk                   int64
Sex                        int64
Age                        int64
Education                  int64
Income                     int64
Diabetes_binary            int64
Age_Groups              category
Income_Groups           category
Education_Groups        category
Sex_Group                 object
dtype: object

There are no missing values to clean. Note that although we have duplicate entries, we leave them for two main reasons:
1. They represent different respondants who happen to have identical responses to all of their survey questions; they are not erroneous records
2. Removing them would would also redice our dataset size greatly and change the rewpresentation of certain responses.

In [12]:
# Now we want to seperate the features from the target
# We want to only keep the original features
X = df.drop(columns=[
    "Diabetes_binary",
    "Sex_Group",
    "Age_Groups",
    "Income_Groups",
    "Education_Groups"
])

# Only the target
y = df["Diabetes_binary"]